# Azure AI Content Understanding — analyze a document

Runs a local document (PDF / image) through an Azure **Content Understanding** analyzer and inspects the result: extracted **markdown**, **pages**, **words / lines with bounding boxes**, **tables**, and **fields** (with confidence).

Built on the Python SDK (`azure-ai-contentunderstanding`), which wraps these REST operations:

- **Analyze** — `POST {endpoint}/contentunderstanding/analyzers/{analyzerId}:analyze?api-version=2025-11-01` → `202` + `Operation-Location` header  
  <https://learn.microsoft.com/en-us/rest/api/contentunderstanding/content-analyzers/analyze?view=rest-contentunderstanding-2025-11-01>
- **Get Result** — `GET {endpoint}/contentunderstanding/analyzerResults/{operationId}?api-version=2025-11-01` (poll until `status: Succeeded`)  
  <https://learn.microsoft.com/en-us/rest/api/contentunderstanding/content-analyzers/get-result?view=rest-contentunderstanding-2025-11-01>

### How positions are encoded
Every element (word, line, table cell, field) carries a `source` string like `D(1, 0.5,0.5, 1.7,0.5, 1.7,0.7, 0.5,0.7)` — the leading `D` = document, then the **1-based page number**, then a **bounding polygon** as `x,y` pairs. Units are **inches** for PDF and **pixels** for images (`content.unit`). Elements also carry `span`/`spans` = character offset + length into the extracted markdown.

> **Note:** words, lines, paragraphs and tables are only returned when the analyzer has OCR / layout **and** `returnDetails` enabled. `prebuilt-documentAnalyzer` returns them by default; a custom analyzer must be configured for it. `fields` are only returned by custom analyzers with a field schema.

## 0. Install dependencies

In [ ]:
%pip install --pre --quiet azure-ai-contentunderstanding
%pip install --quiet azure-identity pandas

## 1. Configuration

The API key is read from the environment variable `AZURE_CU_KEY`. If you'd rather paste it here, set `KEY_FALLBACK` — but **do not commit the notebook with a real key in it**.

In [ ]:
import os
from pathlib import Path

# --- Resource -------------------------------------------------------------
ENDPOINT = "https://<your-resource>.services.ai.azure.com/"
API_VERSION = "2025-11-01"

# Prefer the env var; fall back to a value pasted here (leave "" to force env var).
KEY_FALLBACK = ""
KEY = os.environ.get("AZURE_CU_KEY", "").strip() or KEY_FALLBACK.strip()

# --- What to analyze ----------------------------------------------------
# 'prebuilt-documentAnalyzer' -> OCR + layout + words/lines/tables (no custom fields).
# Swap for your own analyzer id (e.g. 'mytestanalyzer') to also get extracted fields.
ANALYZER_ID = "prebuilt-documentAnalyzer"

# Local document to send (PDF, JPEG, PNG, TIFF, ...). It is uploaded as raw bytes.
DOC_PATH = Path(r"C:\Users\<you>\Downloads\Mu10Q2026.pdf")

assert ENDPOINT.startswith("http"), "Set a valid ENDPOINT"
assert KEY, "No API key found — set the AZURE_CU_KEY env var or KEY_FALLBACK"
assert DOC_PATH.is_file(), f"File not found: {DOC_PATH.resolve()}"
print(f"Analyzer : {ANALYZER_ID}")
print(f"Document : {DOC_PATH.resolve()}  ({DOC_PATH.stat().st_size:,} bytes)")

AssertionError: File not found: C:\Users\<you>\Documents\LearnCode\azure_content_understanding\sample.pdf

## 2. Create the client

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential

credential = AzureKeyCredential(KEY)
# Alternative (no key): from azure.identity import DefaultAzureCredential; credential = DefaultAzureCredential()

client = ContentUnderstandingClient(
    endpoint=ENDPOINT,
    credential=credential,
    api_version=API_VERSION,
)
print("client ready")

## 2b. List analyzers on this resource

Confirms which analyzer IDs actually exist before calling analyze — a wrong `ANALYZER_ID` returns 404. Uses the REST list endpoint directly: `GET {endpoint}/contentunderstanding/analyzers?api-version=...`. If `prebuilt-documentAnalyzer` isn't in the output, set `ANALYZER_ID` (cell 1) to one that is and re-run from there.

In [ ]:
import json, urllib.request

_url = f"{ENDPOINT.rstrip('/')}/contentunderstanding/analyzers?api-version={API_VERSION}"
_req = urllib.request.Request(_url, headers={"Ocp-Apim-Subscription-Key": KEY})
with urllib.request.urlopen(_req) as _resp:
    _data = json.load(_resp)

_analyzers = _data.get("value", _data if isinstance(_data, list) else [])
print(f"{len(_analyzers)} analyzer(s):\n")
for a in _analyzers:
    print(f"  {str(a.get('analyzerId')):40}  status={a.get('status')}  {a.get('description', '')}")

_ids = [a.get("analyzerId") for a in _analyzers]
if _ids and ANALYZER_ID not in _ids:
    print(
        f"\n[!] ANALYZER_ID={ANALYZER_ID!r} is not listed. "
        "If the analyze call 404s, set it to one of the above and re-run from cell 1."
    )

## 3. Analyze the document

`begin_analyze_binary` uploads the file bytes, then the poller waits through `NotStarted → Running → Succeeded` (the SDK does the `Operation-Location` polling for you).

In [ ]:
import json, time

t0 = time.time()
poller = client.begin_analyze_binary(
    analyzer_id=ANALYZER_ID,
    binary_input=DOC_PATH.read_bytes(),
)
result = poller.result()
print(f"done in {time.time() - t0:.1f}s")

res = result.as_dict()

out_path = DOC_PATH.with_name(DOC_PATH.stem + "_result.json")
out_path.write_text(json.dumps(res, indent=2), encoding="utf-8")
print(f"full result JSON -> {out_path.resolve()}")

## 4. Raw result JSON (preview)

In [ ]:
preview = json.dumps(res, indent=2).splitlines()
MAX = 80
print("\n".join(preview[:MAX]))
if len(preview) > MAX:
    print(f"\n... {len(preview) - MAX} more lines (see {out_path.name})")

In [ ]:
contents = res.get("contents", []) or []
print(f"analyzerId : {res.get('analyzerId')}")
print(f"apiVersion : {res.get('apiVersion')}")
print(f"contents   : {len(contents)}")
for i, c in enumerate(contents):
    print(
        f"  [{i}] kind={c.get('kind')} mime={c.get('mimeType')} unit={c.get('unit')} "
        f"pages={c.get('startPageNumber')}-{c.get('endPageNumber')} "
        f"(#pages={len(c.get('pages', []) or [])}, #tables={len(c.get('tables', []) or [])}, "
        f"#fields={len(c.get('fields', {}) or {})})"
    )

## 5. Helper: decode the `source` bounding-polygon string

`D(page, x1,y1, x2,y2, x3,y3, x4,y4)` → page number + polygon + an axis-aligned `(x0,y0,x1,y1)` bounding box.

In [ ]:
def parse_source(source):
    """Decode a Content Understanding `source` string into page + polygon + bbox.

    Returns None if the string is missing or not a document region (e.g. audio/visual).
    """
    if not isinstance(source, str) or not source.startswith("D("):
        return None
    inner = source[2:].rstrip(")")
    parts = [p.strip() for p in inner.split(",") if p.strip() != ""]
    try:
        nums = [float(p) for p in parts]
    except ValueError:
        return None
    if len(nums) < 3:
        return None
    page = int(nums[0])
    xs, ys = nums[1::2], nums[2::2]
    if not xs or not ys:
        return None
    return {
        "page": page,
        "polygon": list(zip(xs, ys)),
        "x0": min(xs), "y0": min(ys),
        "x1": max(xs), "y1": max(ys),
    }


# quick check against the shape shown in the docs
parse_source("D(1, 5,1, 7,1, 7,1.5, 5,1.5)")

## 6. Pages summary

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)

page_rows = []
for ci, c in enumerate(contents):
    for p in c.get("pages", []) or []:
        page_rows.append({
            "content": ci,
            "page": p.get("pageNumber"),
            "width": p.get("width"),
            "height": p.get("height"),
            "unit": c.get("unit"),
            "angle": p.get("angle"),
            "words": len(p.get("words", []) or []),
            "lines": len(p.get("lines", []) or []),
        })
pages_df = pd.DataFrame(page_rows)
pages_df

## 7. Words with bounding boxes

In [ ]:
word_rows = []
for ci, c in enumerate(contents):
    unit = c.get("unit")
    for p in c.get("pages", []) or []:
        for w in p.get("words", []) or []:
            g = parse_source(w.get("source")) or {}
            span = w.get("span") or {}
            word_rows.append({
                "content": ci,
                "page": g.get("page", p.get("pageNumber")),
                "text": w.get("content"),
                "confidence": w.get("confidence"),
                "x0": g.get("x0"), "y0": g.get("y0"),
                "x1": g.get("x1"), "y1": g.get("y1"),
                "unit": unit,
                "span_offset": span.get("offset"),
                "span_length": span.get("length"),
                "source": w.get("source"),
            })
words_df = pd.DataFrame(word_rows)
print(f"{len(words_df)} words")
words_df.head(50)

In [ ]:
# lowest-confidence words — handy for spotting OCR trouble
if not words_df.empty and words_df["confidence"].notna().any():
    display(words_df.sort_values("confidence").head(15))
else:
    print("no per-word confidence in this result")

## 8. Lines with bounding boxes

In [ ]:
line_rows = []
for ci, c in enumerate(contents):
    unit = c.get("unit")
    for p in c.get("pages", []) or []:
        for ln in p.get("lines", []) or []:
            g = parse_source(ln.get("source")) or {}
            span = ln.get("span") or {}
            line_rows.append({
                "content": ci,
                "page": g.get("page", p.get("pageNumber")),
                "text": ln.get("content"),
                "x0": g.get("x0"), "y0": g.get("y0"),
                "x1": g.get("x1"), "y1": g.get("y1"),
                "unit": unit,
                "span_offset": span.get("offset"),
                "span_length": span.get("length"),
            })
lines_df = pd.DataFrame(line_rows)
print(f"{len(lines_df)} lines")
lines_df.head(50)

## 9. Tables

In [ ]:
from IPython.display import display

any_tables = False
for ci, c in enumerate(contents):
    for ti, t in enumerate(c.get("tables", []) or []):
        any_tables = True
        print(f"content {ci} — table {ti}: {t.get('rowCount')} rows x {t.get('columnCount')} cols")
        cell_rows = []
        for cell in t.get("cells", []) or []:
            g = parse_source(cell.get("source")) or {}
            cell_rows.append({
                "row": cell.get("rowIndex"), "col": cell.get("columnIndex"),
                "rowSpan": cell.get("rowSpan"), "colSpan": cell.get("columnSpan"),
                "kind": cell.get("kind"),
                "content": cell.get("content"),
                "page": g.get("page"),
                "x0": g.get("x0"), "y0": g.get("y0"),
                "x1": g.get("x1"), "y1": g.get("y1"),
            })
        display(pd.DataFrame(cell_rows))

        # reshape into a grid (ignoring spans) for a readable preview
        grid = [[""] * (t.get("columnCount") or 0) for _ in range(t.get("rowCount") or 0)]
        for cell in t.get("cells", []) or []:
            r, cc = cell.get("rowIndex"), cell.get("columnIndex")
            if r is not None and cc is not None and r < len(grid) and cc < len(grid[0]):
                grid[r][cc] = cell.get("content", "")
        display(pd.DataFrame(grid))

if not any_tables:
    print("no tables in this result (needs an analyzer with layout + returnDetails)")

## 10. Extracted fields

Populated by **custom analyzers** with a field schema. Nested `object` / `array` fields are flattened with dotted / indexed names.

In [ ]:
def flatten_fields(fields, prefix=""):
    rows = []
    for name, f in (fields or {}).items():
        if not isinstance(f, dict):
            continue
        key = f"{prefix}{name}"
        ftype = f.get("type")
        value = next((v for k, v in f.items() if k.startswith("value")), None)
        g = parse_source(f.get("source")) or {}
        spans = f.get("spans") or ([f["span"]] if f.get("span") else [])
        rows.append({
            "field": key,
            "type": ftype,
            "value": f"<{ftype}>" if ftype in ("array", "object") else value,
            "confidence": f.get("confidence"),
            "page": g.get("page"),
            "x0": g.get("x0"), "y0": g.get("y0"),
            "x1": g.get("x1"), "y1": g.get("y1"),
            "spans": spans,
        })
        if ftype == "object":
            rows += flatten_fields(f.get("valueObject"), prefix=key + ".")
        elif ftype == "array":
            for i, item in enumerate(f.get("valueArray") or []):
                rows += flatten_fields({f"[{i}]": item}, prefix=key)
    return rows


field_rows = []
for ci, c in enumerate(contents):
    for r in flatten_fields(c.get("fields", {})):
        r = {"content": ci, **r}
        field_rows.append(r)

fields_df = pd.DataFrame(field_rows)
if fields_df.empty:
    print("no fields in this result (expected for prebuilt-documentAnalyzer)")
fields_df

## 11. Extracted markdown

In [ ]:
for ci, c in enumerate(contents):
    md = c.get("markdown") or ""
    print(f"===== content {ci} markdown ({len(md)} chars) =====\n")
    print(md[:4000])
    if len(md) > 4000:
        print(f"\n... {len(md) - 4000} more chars")

In [ ]:
# Render it (first content) as actual formatted markdown
from IPython.display import Markdown

Markdown(contents[0].get("markdown", "")) if contents else "no content"